In [1]:
import os
import json
from dotenv import load_dotenv
from scraper import fetch_website_contents,fetch_website_links
from IPython.display import Markdown,display, update_display
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()





link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

        
def select_relevant_links(url):
    print(f"selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(model=MODEL, messages=[{"role":"system","content": link_system_prompt}, {"role": "user", "content": get_links_user_prompt(url)}], response_format = {"type":"json_object"})
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"found {len(links["links"])} relevant link")
    return links


def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result



brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt



def create_brochure(company_name, url):
    response = openai.chat.completions.create(model="gpt-5-nano", messages = [{"role": "system", "content":brochure_system_prompt}, {"role":"user", "content": get_brochure_user_prompt(company_name, url)}])
    result = response.choices[0].message.content
    display(Markdown(result))



def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

stream_brochure("learn2earn", "https://learn2earn.ng/")

API key looks good so far
selecting relevant links for https://learn2earn.ng/ by calling gpt-5-nano
found 0 relevant link


# Learn2Earn: Learn to Code, Earn While Learning

---

## About Learn2Earn

Learn2Earn is a cutting-edge AI-focused digital skills training program designed to launch your tech career in the AI era. With a mission to build an AI-native future, Learn2Earn offers full-stack engineering education that goes beyond traditional coding — integrating AI, emerging technologies, and real-world experience from day one.

---

## Program Highlights

- **Zero Upfront Tuition:** World-class training accessible to everyone with no upfront costs and no student debt.
- **Inclusive Learning:** No prior coding or tech background required.
- **2 Years of Immersive Access:** 1 year of intensive training and 1 year of access to paid work.
- **AI-First Engineering:** Learn to build, adapt, and innovate with AI and the latest technologies.
- **Real Projects & Experience:** Learn by building on real-world projects using in-demand tools and AI.
- **Paid Work Opportunities:** Gain certification and direct connections to paid roles at global tech companies.
- **On-Campus Experience:** Learn in-person at state-of-the-art campuses across Nigeria, with every learner assigned a laptop.

---

## The Learn2Earn Journey

1. **Play the Selection Game:** A free, 90-minute online game designed to test memory and problem-solving skills — no prior knowledge needed.
2. **Trials:** A 5-day immersive challenge for those who qualify.
3. **Immersive Tech Training:** Full-stack AI-native engineering over one year.
4. **Paid Work Access:** One year of paid opportunities working with global tech employers.

Secure a spot through the limited application slots opened across Nigeria.

---

## Our Partners

Learn2Earn collaborates with international leaders such as **Local Globe** to bridge local talent with the global tech ecosystem.

---

## Company Culture

- **Innovative & Future-Focused:** At Learn2Earn, innovation is at the core, empowering learners to be ahead in the AI era.
- **Inclusive & Accessible:** Education without barriers — no tuition fees, no prior background required.
- **Collaborative & Immersive:** Hands-on learning together in vibrant on-campus environments.
- **Driven by Impact:** Focused on transforming lives and creating global opportunities for Nigerian tech talent.

---

## Careers and Opportunities

Join Learn2Earn if you are passionate about making an impact through technology and AI. The program is ideal for individuals seeking to:

- Launch or pivot into tech careers without prior experience.
- Gain hands-on skills with real AI tools used by industry leaders.
- Access international paid job roles with top-tier certification.
  
---

## Join Learn2Earn Today

- **Who:** Aspiring engineers and AI natives across Nigeria.
- **What:** Full-stack AI engineering training with paid work experience.
- **Where:** State-of-the-art campuses across Nigeria; remote elements included.
- **How:** Apply now by playing the **Selection Game** online.

**Take the first step toward your AI-powered career. Learn, build, and earn with Learn2Earn!**

---

For more information and to apply, visit: [Learn2Earn Website](#)  
Email: info@learn2earn.com  
Phone: +234-XXX-XXX-XXXX

---

*Learn2Earn – Empowering Nigeria’s future engineers in the AI era.*